# 🎯 Week 1: Imitation Learning & Behavioral Cloning

**Mapped to:** CS285 Lectures 2 (Supervised Learning of Behaviors) & HW1 (Imitation Learning)  
**Reference:** [UC Berkeley CS285 Fall 2023](https://rail.eecs.berkeley.edu/deeprlcourse-fa23/)  
**Instructor:** Sergey Levine  

---

### What You'll Learn & Implement

| Section | Topic | Why It Matters |
|---------|-------|----------------|
| §1 | **MDP Foundations** | The language of all RL — states, actions, rewards, transitions |
| §2 | **Behavioral Cloning (BC)** | Simplest imitation — supervised learning on expert demos |
| §3 | **Distribution Shift Problem** | Why BC fails — the compounding error insight |
| §4 | **DAgger Algorithm** | The fix — aggregate data from the learned policy |
| §5 | **BC vs DAgger Experiments** | Head-to-head comparison with analysis |
| §6 | **Key Takeaways & Bridge to Week 2** | What this means for RLHF later |

> **⏱ Estimated time: 10-12 hours** (watch lectures + run notebook + write reflections)

---
## 📦 Setup & Installation

We use **Gymnasium** (successor to OpenAI Gym) with classic control environments.  
No MuJoCo license needed — we use `CartPole-v1` and `LunarLander-v2` for clean, interpretable experiments.

In [1]:
# Install dependencies
!pip install gymnasium[classic_control] torch numpy matplotlib tqdm ipywidgets -q
!pip install gymnasium[box2d] -q  # For LunarLander

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

zsh:1: no matches found: gymnasium[classic_control]
zsh:1: no matches found: gymnasium[box2d]


ModuleNotFoundError: No module named 'gymnasium'

---
## §1. MDP Foundations — The Language of RL

📖 **CS285 Reference:** Lecture 4 (Introduction to RL) — but we need this vocabulary now.

### Key Concepts

A **Markov Decision Process (MDP)** is defined by the tuple $(\mathcal{S}, \mathcal{A}, P, r, \gamma)$:

| Symbol | Meaning | CartPole Example |
|--------|---------|------------------|
| $\mathcal{S}$ | State space | [cart_pos, cart_vel, pole_angle, pole_angular_vel] |
| $\mathcal{A}$ | Action space | {0: push_left, 1: push_right} |
| $P(s'\|s,a)$ | Transition dynamics | Physics simulation |
| $r(s,a)$ | Reward function | +1 for every step pole stays up |
| $\gamma$ | Discount factor | How much we value future vs present rewards |

A **policy** $\pi(a|s)$ maps states to action distributions. Our goal:

$$\pi^* = \arg\max_\pi \mathbb{E}_{\tau \sim \pi} \left[ \sum_{t=0}^{T} \gamma^t r(s_t, a_t) \right]$$

### The Markov Property
The future is independent of the past given the present:  
$P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} | s_t, a_t)$

**Why this matters:** This is what makes RL tractable. Without it, we'd need full history — which is actually what transformers give us in the LLM setting (foreshadowing Decision Transformer in Week 9).

In [ ]:
# Let's explore our environments and understand the MDP structure

def explore_environment(env_name):
    """Understand the MDP structure of a Gymnasium environment."""
    env = gym.make(env_name)
    print(f"\n{'='*60}")
    print(f"Environment: {env_name}")
    print(f"{'='*60}")
    print(f"\nState space (S):  {env.observation_space}")
    print(f"  Shape: {env.observation_space.shape}")
    if hasattr(env.observation_space, 'low'):
        print(f"  Low:   {env.observation_space.low}")
        print(f"  High:  {env.observation_space.high}")
    print(f"\nAction space (A): {env.action_space}")
    if hasattr(env.action_space, 'n'):
        print(f"  Discrete with {env.action_space.n} actions")

    # Run one episode to see the dynamics
    obs, info = env.reset(seed=SEED)
    print(f"\nInitial state s_0: {obs}")

    total_reward = 0
    for step in range(5):
        action = env.action_space.sample()  # Random policy
        next_obs, reward, terminated, truncated, info = env.step(action)
        print(f"  Step {step}: a={action}, r={reward:.2f}, s'={next_obs[:3]}..., done={terminated}")
        total_reward += reward
        if terminated:
            break
        obs = next_obs

    env.close()
    return

explore_environment('CartPole-v1')
explore_environment('LunarLander-v2')

### 💡 Key Insight #1: State Representation Matters

Notice that CartPole gives us a **4-dimensional continuous state** while the action is **discrete (2 choices)**. This is a common pattern:
- The state contains all the information needed to predict the future (Markov property)
- The policy must map continuous states → discrete actions
- This is why neural networks are useful — they can approximate arbitrary mappings

**Connection to LLMs:** In RLHF (Week 11), the "state" is the prompt + generated tokens so far, and the "action" is the next token from a vocabulary of 32K+. Same framework, much larger spaces.

---
## §2. Building an Expert & Behavioral Cloning

📖 **CS285 Reference:** Lecture 2 — Supervised Learning of Behaviors

### The Imitation Learning Setup

We have an **expert** $\pi^*$ that can solve the task. We want to learn a policy $\pi_\theta$ that imitates it.

**Behavioral Cloning (BC)** treats this as supervised learning:
1. Collect dataset $\mathcal{D} = \{(s_i, a_i^*)\}$ from expert demonstrations
2. Train $\pi_\theta$ to minimize: $\mathcal{L}(\theta) = \mathbb{E}_{(s,a^*) \sim \mathcal{D}} \left[ -\log \pi_\theta(a^* | s) \right]$

That's it — just cross-entropy loss, like any classification problem.

### Why Study This?
- BC is the **simplest possible approach** to learning from demonstrations
- Understanding why it fails motivates everything else in RL
- **SFT in LLMs is literally BC** — we train on (prompt, expert_response) pairs

In [ ]:
# ===========================================================
# STEP 1: Create an Expert Policy
# ===========================================================
# We'll train an expert using a simple hand-crafted heuristic for CartPole
# and a trained neural net for LunarLander

class CartPoleExpert:
    """
    Hand-crafted expert for CartPole-v1.
    Uses the pole angle and angular velocity to decide.
    This is a perfect expert — achieves max reward of 500.
    """
    def predict(self, obs):
        # obs: [cart_pos, cart_vel, pole_angle, pole_angular_vel]
        pole_angle = obs[2]
        pole_velocity = obs[3]
        # Simple PD controller: push right if pole is leaning/falling right
        if pole_angle + 0.25 * pole_velocity > 0:
            return 1  # Push right
        else:
            return 0  # Push left


def evaluate_policy(policy_fn, env_name, n_episodes=20, max_steps=500, render=False):
    """Evaluate a policy and return mean/std reward."""
    env = gym.make(env_name)
    rewards = []
    episode_lengths = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=SEED + ep)
        total_reward = 0
        for step in range(max_steps):
            action = policy_fn(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        rewards.append(total_reward)
        episode_lengths.append(step + 1)

    env.close()
    return np.mean(rewards), np.std(rewards), np.mean(episode_lengths)


# Evaluate our expert
expert = CartPoleExpert()
mean_r, std_r, mean_len = evaluate_policy(expert.predict, 'CartPole-v1')
print(f"Expert performance: {mean_r:.1f} ± {std_r:.1f} reward, {mean_len:.0f} avg steps")
print(f"(Max possible: 500 — the episode truncates at 500 steps)")

In [ ]:
# ===========================================================
# STEP 2: Collect Expert Demonstrations
# ===========================================================

def collect_demonstrations(expert_fn, env_name, n_episodes, max_steps=500, seed=SEED):
    """
    Collect (state, action) pairs from an expert policy.
    This is the dataset D = {(s_i, a*_i)} for behavioral cloning.
    """
    env = gym.make(env_name)
    states, actions = [], []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        for step in range(max_steps):
            action = expert_fn(obs)
            states.append(obs.copy())
            actions.append(action)
            obs, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break

    env.close()
    states = np.array(states, dtype=np.float32)
    actions = np.array(actions, dtype=np.int64)
    print(f"Collected {len(states)} transitions from {n_episodes} episodes")
    print(f"State shape: {states.shape}, Action distribution: {np.bincount(actions)}")
    return states, actions


# Collect different dataset sizes — we'll study the effect later
demos_small = collect_demonstrations(expert.predict, 'CartPole-v1', n_episodes=5)
demos_medium = collect_demonstrations(expert.predict, 'CartPole-v1', n_episodes=25)
demos_large = collect_demonstrations(expert.predict, 'CartPole-v1', n_episodes=100)

In [ ]:
# ===========================================================
# STEP 3: Define the Policy Network (π_θ)
# ===========================================================

class PolicyNetwork(nn.Module):
    """
    A simple MLP policy: s → π_θ(a|s)

    Architecture decisions:
    - 2 hidden layers with ReLU (standard for control tasks)
    - Output = logits over actions (not probabilities)
    - We apply softmax only when sampling; cross-entropy loss expects logits
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, x):
        """Returns logits (unnormalized log-probabilities)."""
        return self.net(x)

    def get_action(self, obs):
        """Select action greedily (argmax) for evaluation."""
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            logits = self.forward(obs_t)
            return logits.argmax(dim=-1).item()

    def get_action_stochastic(self, obs):
        """Sample action from the policy distribution."""
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            logits = self.forward(obs_t)
            probs = torch.softmax(logits, dim=-1)
            return torch.multinomial(probs, 1).item()


# Quick sanity check
policy = PolicyNetwork(obs_dim=4, act_dim=2).to(device)
print(f"Policy architecture:\n{policy}")
print(f"\nTotal parameters: {sum(p.numel() for p in policy.parameters()):,}")

# Test with a random observation
test_obs = np.array([0.01, -0.02, 0.03, 0.04], dtype=np.float32)
print(f"\nTest obs: {test_obs}")
print(f"Greedy action: {policy.get_action(test_obs)}")
print(f"Stochastic action: {policy.get_action_stochastic(test_obs)}")

In [ ]:
# ===========================================================
# STEP 4: Behavioral Cloning Training Loop
# ===========================================================

def train_behavioral_cloning(states, actions, obs_dim, act_dim,
                              hidden_dim=64, lr=1e-3, n_epochs=50,
                              batch_size=64, eval_env='CartPole-v1',
                              eval_every=10, verbose=True):
    """
    Train a policy via Behavioral Cloning.

    This is pure supervised learning:
      Loss = CrossEntropy(π_θ(s), a*)

    Equivalent to minimizing: -E[(s,a*)~D] [log π_θ(a*|s)]
    which is the negative log-likelihood of expert actions.
    """
    # Create policy and optimizer
    policy = PolicyNetwork(obs_dim, act_dim, hidden_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()  # Expects logits, not probabilities!

    # Create DataLoader
    dataset = TensorDataset(
        torch.FloatTensor(states).to(device),
        torch.LongTensor(actions).to(device)
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Training history
    history = {'loss': [], 'eval_reward_mean': [], 'eval_reward_std': []}

    for epoch in range(n_epochs):
        # Training
        policy.train()
        epoch_losses = []
        for batch_states, batch_actions in dataloader:
            logits = policy(batch_states)
            loss = loss_fn(logits, batch_actions)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        avg_loss = np.mean(epoch_losses)
        history['loss'].append(avg_loss)

        # Evaluation
        if (epoch + 1) % eval_every == 0 or epoch == 0:
            policy.eval()
            mean_r, std_r, _ = evaluate_policy(
                policy.get_action, eval_env, n_episodes=20
            )
            history['eval_reward_mean'].append(mean_r)
            history['eval_reward_std'].append(std_r)
            if verbose:
                print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | "
                      f"Eval Reward: {mean_r:.1f} ± {std_r:.1f}")

    return policy, history


print("Training BC with MEDIUM dataset (25 expert episodes)...")
print("="*60)
bc_policy, bc_history = train_behavioral_cloning(
    *demos_medium, obs_dim=4, act_dim=2,
    n_epochs=100, eval_every=20
)

In [ ]:
# ===========================================================
# STEP 5: Visualize Training
# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(bc_history['loss'], color='#2E5090', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('BC Training Loss', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Reward curve
eval_epochs = [0] + list(range(19, 100, 20))
means = bc_history['eval_reward_mean']
stds = bc_history['eval_reward_std']
axes[1].plot(eval_epochs[:len(means)], means, color='#2D8B55', linewidth=2, marker='o')
axes[1].fill_between(eval_epochs[:len(means)],
                      [m-s for m,s in zip(means, stds)],
                      [m+s for m,s in zip(means, stds)],
                      alpha=0.2, color='#2D8B55')
axes[1].axhline(y=500, color='red', linestyle='--', alpha=0.7, label='Expert (500)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Mean Episode Reward', fontsize=12)
axes[1].set_title('BC Evaluation Performance', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## §3. The Distribution Shift Problem — Why BC Fails

📖 **CS285 Reference:** Lecture 2, slides on "distributional shift"

### The Core Problem

Behavioral cloning trains on states visited by the **expert** $p_{\pi^*}(s)$, but at test time the **learned policy** visits different states $p_{\pi_\theta}(s)$.

Even a small error $\epsilon$ per step compounds:

$$\text{Total error} \propto \epsilon T^2$$

where $T$ is the episode length. This is the **quadratic compounding error** — the defining problem of behavioral cloning.

### Intuition
Imagine learning to drive by watching an expert:
- The expert always drives in the center of the lane
- You learn: "when in center → steer straight"
- At test time, a small error puts you slightly off-center
- **You've never seen this state!** Your policy has no idea what to do
- You drift further... compounding errors... crash

### Mathematical Formalization
The BC objective optimizes:
$$\min_\theta \mathbb{E}_{s \sim p_{\pi^*}(s)} [\ell(\pi_\theta(s), \pi^*(s))]$$

But what we actually care about is performance under $p_{\pi_\theta}(s)$:
$$\min_\theta \mathbb{E}_{s \sim p_{\pi_\theta}(s)} [\ell(\pi_\theta(s), \pi^*(s))]$$

When $p_{\pi^*}(s) \neq p_{\pi_\theta}(s)$, we have a **distribution mismatch**.

### 🔗 Connection to LLMs
This exact problem shows up in SFT for LLMs!
- Training: model sees human-written completions (expert distribution)
- Inference: model generates its own tokens, drifting into states it never trained on
- This is one reason why RLHF (Week 11) was invented — it trains the model on its OWN outputs

In [ ]:
# ===========================================================
# EXPERIMENT: Demonstrate Distribution Shift
# ===========================================================
# We'll show that BC's learned policy visits states the expert never did

def collect_state_distribution(policy_fn, env_name, n_episodes=50,
                                max_steps=500, seed=SEED):
    """Collect the distribution of states visited by a policy."""
    env = gym.make(env_name)
    all_states = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        for step in range(max_steps):
            all_states.append(obs.copy())
            action = policy_fn(obs)
            obs, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
    env.close()
    return np.array(all_states)


# Collect state distributions
expert_states = collect_state_distribution(expert.predict, 'CartPole-v1')
bc_states = collect_state_distribution(bc_policy.get_action, 'CartPole-v1')

# Visualize the distribution shift
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
state_names = ['Cart Position', 'Cart Velocity', 'Pole Angle', 'Pole Angular Velocity']

for i, (ax, name) in enumerate(zip(axes.flat, state_names)):
    ax.hist(expert_states[:, i], bins=50, alpha=0.6, color='#2D8B55',
            label='Expert', density=True, edgecolor='white')
    ax.hist(bc_states[:, i], bins=50, alpha=0.6, color='#C0392B',
            label='BC Policy', density=True, edgecolor='white')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Distribution Shift: Expert vs BC Policy State Distributions',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 State distribution statistics:")
print(f"Expert states collected: {len(expert_states)}")
print(f"BC policy states collected: {len(bc_states)}")
print(f"\nIf BC is shorter → it failed early (compounding errors!)")
print(f"Notice how BC visits more extreme states the expert never saw.")

In [ ]:
# ===========================================================
# EXPERIMENT: Effect of Dataset Size on BC Performance
# ===========================================================
# More data helps but can't fully solve distribution shift

dataset_sizes = [1, 3, 5, 10, 25, 50, 100]
bc_results = []

print("Training BC with varying dataset sizes...")
print("="*60)

for n_eps in dataset_sizes:
    states, actions = collect_demonstrations(
        expert.predict, 'CartPole-v1', n_episodes=n_eps
    )
    policy, _ = train_behavioral_cloning(
        states, actions, obs_dim=4, act_dim=2,
        n_epochs=100, eval_every=100, verbose=False
    )
    mean_r, std_r, _ = evaluate_policy(policy.get_action, 'CartPole-v1', n_episodes=30)
    bc_results.append((n_eps, mean_r, std_r))
    print(f"  {n_eps:3d} expert episodes ({len(states):5d} transitions) → "
          f"Reward: {mean_r:.1f} ± {std_r:.1f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
sizes = [r[0] for r in bc_results]
means = [r[1] for r in bc_results]
stds = [r[2] for r in bc_results]

ax.errorbar(sizes, means, yerr=stds, color='#2E5090', linewidth=2,
            marker='o', markersize=8, capsize=5, label='Behavioral Cloning')
ax.axhline(y=500, color='red', linestyle='--', alpha=0.7, label='Expert (500)')
ax.set_xlabel('Number of Expert Episodes', fontsize=12)
ax.set_ylabel('Mean Evaluation Reward', fontsize=12)
ax.set_title('BC Performance vs Dataset Size\n(More data helps, but distribution shift remains)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

### 💡 Key Insight #2: Data Alone Doesn't Fix Distribution Shift

Even with 100 expert episodes (50,000 transitions), BC may not match the expert perfectly. The fundamental issue isn't data quantity — it's that BC never trains on the states its own policy visits.

**Three ways to address this (CS285 Lecture 2):**
1. **DAgger** — collect data from the learned policy, label with expert → we'll implement this next
2. **Be very smart about the policy class** — make it hard to make mistakes (e.g., output corrections not actions)
3. **Do RL instead** — optimize for actual reward, training on your own state distribution → Week 2 onwards

---
## §4. DAgger: Dataset Aggregation

📖 **CS285 Reference:** Lecture 2, HW1

**DAgger** (Ross et al., 2011) directly addresses distribution shift:

### Algorithm
```
1. Train π_θ on initial expert data D = {(s, a*)} via BC
2. For iteration i = 1, 2, ...:
   a. Run π_θ in environment → collect states {s_1, s_2, ...}
   b. Query expert for labels on these states → {(s_j, π*(s_j))}
   c. Aggregate: D = D ∪ {new labeled data}
   d. Retrain π_θ on full aggregated dataset D
```

### Why It Works
- In step 2a, we collect states from $p_{\pi_\theta}(s)$ — the learned policy's distribution
- In step 2b, the expert tells us what to do in those states
- Over iterations, $\mathcal{D}$ covers both $p_{\pi^*}(s)$ AND $p_{\pi_\theta}(s)$
- The distribution mismatch shrinks!

### Theoretical Guarantee
DAgger achieves regret linear in $T$ (instead of quadratic for BC):
$$J(\pi^*) - J(\pi_\theta) \leq \epsilon T + O(1)$$

### ⚠️ The Catch
DAgger requires **querying the expert** at every iteration. In robotics, this means a human labeling actions. In LLMs, this means getting human preference labels — which is expensive, motivating RLAIF and DPO (Weeks 13-18).

In [ ]:
# ===========================================================
# DAgger Implementation
# ===========================================================

def train_dagger(expert_fn, env_name, obs_dim, act_dim,
                  n_initial_episodes=5, n_dagger_iterations=10,
                  n_rollout_episodes=5, max_steps=500,
                  n_train_epochs=50, hidden_dim=64, lr=1e-3,
                  batch_size=64, mixing_ratio=0.0, verbose=True):
    """
    DAgger: Dataset Aggregation for Imitation Learning.

    Args:
        mixing_ratio: β in [0,1]. Probability of using expert (vs learned policy)
                      during rollouts. β=0 → pure learned policy. β=1 → pure expert.
                      Classic DAgger uses β=0 after first iteration.
    """
    env = gym.make(env_name)

    # Step 1: Collect initial expert demonstrations
    all_states, all_actions = [], []
    for ep in range(n_initial_episodes):
        obs, _ = env.reset(seed=SEED + ep)
        for step in range(max_steps):
            action = expert_fn(obs)
            all_states.append(obs.copy())
            all_actions.append(action)
            obs, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break

    # Initial BC training
    policy = PolicyNetwork(obs_dim, act_dim, hidden_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    history = {
        'iteration': [], 'dataset_size': [],
        'eval_reward_mean': [], 'eval_reward_std': [],
        'train_loss': []
    }

    for dagger_iter in range(n_dagger_iterations + 1):
        # Train policy on aggregated dataset
        states_t = torch.FloatTensor(np.array(all_states)).to(device)
        actions_t = torch.LongTensor(np.array(all_actions)).to(device)
        dataset = TensorDataset(states_t, actions_t)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        policy.train()
        final_loss = 0
        for epoch in range(n_train_epochs):
            epoch_loss = 0
            for batch_s, batch_a in dataloader:
                logits = policy(batch_s)
                loss = loss_fn(logits, batch_a)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            final_loss = epoch_loss / len(dataloader)

        # Evaluate
        policy.eval()
        mean_r, std_r, _ = evaluate_policy(
            policy.get_action, env_name, n_episodes=20
        )
        history['iteration'].append(dagger_iter)
        history['dataset_size'].append(len(all_states))
        history['eval_reward_mean'].append(mean_r)
        history['eval_reward_std'].append(std_r)
        history['train_loss'].append(final_loss)

        if verbose:
            tag = "BC" if dagger_iter == 0 else f"DAgger-{dagger_iter}"
            print(f"  [{tag:10s}] Dataset: {len(all_states):5d} | "
                  f"Loss: {final_loss:.4f} | Reward: {mean_r:.1f} ± {std_r:.1f}")

        # DAgger data collection (skip on last iteration)
        if dagger_iter < n_dagger_iterations:
            for ep in range(n_rollout_episodes):
                obs, _ = env.reset(seed=SEED + 1000 + dagger_iter * 100 + ep)
                for step in range(max_steps):
                    # Roll out with learned policy (or mix)
                    if np.random.random() < mixing_ratio:
                        rollout_action = expert_fn(obs)
                    else:
                        rollout_action = policy.get_action(obs)

                    # KEY STEP: Label with expert, regardless of who acted
                    expert_action = expert_fn(obs)

                    all_states.append(obs.copy())
                    all_actions.append(expert_action)  # Expert label!

                    obs, _, terminated, truncated, _ = env.step(rollout_action)
                    if terminated or truncated:
                        break

    env.close()
    return policy, history


print("Training DAgger (5 initial episodes, 10 iterations)...")
print("="*60)
dagger_policy, dagger_history = train_dagger(
    expert.predict, 'CartPole-v1', obs_dim=4, act_dim=2,
    n_initial_episodes=5, n_dagger_iterations=10,
    n_rollout_episodes=5, n_train_epochs=50
)

---
## §5. BC vs DAgger — Head-to-Head Comparison

In [ ]:
# ===========================================================
# EXPERIMENT: BC vs DAgger at matched dataset sizes
# ===========================================================

# Train BC with the same total amount of data DAgger used
total_dagger_data = dagger_history['dataset_size'][-1]
# Approximate: how many expert episodes give us this many transitions?
approx_episodes = total_dagger_data // 500 + 1

print(f"DAgger used {total_dagger_data} total transitions")
print(f"Training BC with ~{approx_episodes} expert episodes for fair comparison...")
print("="*60)

states_fair, actions_fair = collect_demonstrations(
    expert.predict, 'CartPole-v1', n_episodes=approx_episodes
)
bc_fair_policy, bc_fair_history = train_behavioral_cloning(
    states_fair, actions_fair, obs_dim=4, act_dim=2,
    n_epochs=100, eval_every=100, verbose=False
)
bc_fair_mean, bc_fair_std, _ = evaluate_policy(
    bc_fair_policy.get_action, 'CartPole-v1', n_episodes=30
)
print(f"BC (matched data):  {bc_fair_mean:.1f} ± {bc_fair_std:.1f}")
print(f"DAgger (final):     {dagger_history['eval_reward_mean'][-1]:.1f} ± "
      f"{dagger_history['eval_reward_std'][-1]:.1f}")
print(f"Expert:             500.0 ± 0.0")

In [ ]:
# ===========================================================
# Visualization: The Full Picture
# ===========================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: DAgger learning curve
iters = dagger_history['iteration']
means = dagger_history['eval_reward_mean']
stds = dagger_history['eval_reward_std']
axes[0].plot(iters, means, color='#6C3483', linewidth=2, marker='s', label='DAgger')
axes[0].fill_between(iters, [m-s for m,s in zip(means, stds)],
                     [m+s for m,s in zip(means, stds)], alpha=0.2, color='#6C3483')
axes[0].axhline(y=500, color='red', linestyle='--', alpha=0.7, label='Expert')
axes[0].axhline(y=bc_fair_mean, color='#2E5090', linestyle=':', alpha=0.7,
                label=f'BC (same data): {bc_fair_mean:.0f}')
axes[0].set_xlabel('DAgger Iteration', fontsize=12)
axes[0].set_ylabel('Mean Reward', fontsize=12)
axes[0].set_title('DAgger Convergence', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Dataset growth
axes[1].plot(iters, dagger_history['dataset_size'], color='#E8792B',
             linewidth=2, marker='o')
axes[1].set_xlabel('DAgger Iteration', fontsize=12)
axes[1].set_ylabel('Dataset Size (transitions)', fontsize=12)
axes[1].set_title('DAgger Dataset Growth', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Plot 3: State coverage comparison
dagger_states = collect_state_distribution(dagger_policy.get_action, 'CartPole-v1')
# Compare pole angle distribution (most telling dimension)
axes[2].hist(expert_states[:, 2], bins=50, alpha=0.5, color='#2D8B55',
             label='Expert', density=True, edgecolor='white')
axes[2].hist(bc_states[:, 2], bins=50, alpha=0.5, color='#C0392B',
             label='BC', density=True, edgecolor='white')
axes[2].hist(dagger_states[:, 2], bins=50, alpha=0.5, color='#6C3483',
             label='DAgger', density=True, edgecolor='white')
axes[2].set_xlabel('Pole Angle', fontsize=12)
axes[2].set_ylabel('Density', fontsize=12)
axes[2].set_title('Pole Angle Distribution\n(DAgger matches Expert better)',
                   fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## §6. Key Takeaways & Conceptual Bridges

### ✅ What We Proved This Week

| Concept | Experiment | Result |
|---------|-----------|--------|
| BC works as basic imitation | Trained on 25 expert episodes | Achieves reasonable but imperfect performance |
| Distribution shift is real | Compared state distributions | BC visits states expert never saw |
| More data helps but has limits | Varied dataset size (1→100 eps) | Diminishing returns; shift remains |
| DAgger fixes distribution shift | Compared BC vs DAgger (matched data) | DAgger converges closer to expert |

### 🧠 Deep Conceptual Insights

**1. The Train/Test Mismatch is Fundamental**  
This isn't a bug — it's a fundamental property of sequential decision-making. Any method that trains on one distribution and tests on another will suffer. This is why online learning (RL) exists.

**2. DAgger = Interactive Learning**  
DAgger works because it's *interactive* — it queries the expert on the policy's own states. This is the seed idea behind RLHF: we collect human feedback on the model's own outputs, not just on pre-existing data.

**3. Expert Queries Are Expensive**  
DAgger needs the expert at every iteration. In LLMs:
- Expert = human labelers ranking outputs ($1+ per comparison)
- This cost motivated: DPO (learn from static preferences), RLAIF (use AI as expert), Constitutional AI (use principles)

**4. BC is SFT, DAgger is Online RLHF**  
The parallel is exact:

| Imitation Learning | LLM Training |
|---|---|
| Behavioral Cloning | Supervised Fine-Tuning (SFT) |
| Distribution shift | Exposure bias / train-test mismatch |
| DAgger | Online RLHF (collect feedback on model's own outputs) |
| Expert query cost | Human labeling cost |
| DAgger mixing ratio β | KL penalty in RLHF (stay close to SFT model) |

### 🔜 Bridge to Week 2: Policy Gradients

Imitation learning requires an expert. But what if we don't have one? What if we only have a reward signal?

**Week 2** introduces **REINFORCE** — the simplest policy gradient algorithm:
- Instead of imitating an expert, we optimize the policy directly for reward
- No expert needed — just environment interaction
- But new challenges: high variance, credit assignment, exploration

The progression: **BC (supervised) → DAgger (interactive supervised) → Policy Gradients (RL)**

---
## 📝 Self-Assessment Checklist

Before moving to Week 2, make sure you can answer these:

**Conceptual (explain to a colleague):**
- [ ] What is an MDP? Define each component with a concrete example.
- [ ] Why does behavioral cloning fail? Derive the $\epsilon T^2$ compounding error.
- [ ] How does DAgger fix distribution shift? What's its theoretical guarantee?
- [ ] Why is DAgger impractical for LLMs? What's the cost model?
- [ ] Draw the parallel: BC↔SFT, DAgger↔Online RLHF, expert query↔human labeling.

**Implementation (write from scratch):**
- [ ] Implement a policy network that outputs action probabilities.
- [ ] Write a data collection loop (expert demonstrations).
- [ ] Train BC with cross-entropy loss.
- [ ] Implement DAgger with dataset aggregation.
- [ ] Evaluate and compare policies fairly.

**CS285 Alignment:**
- [ ] Watch Lecture 2: Supervised Learning of Behaviors
- [ ] Read HW1 problem statement (you'll implement the MuJoCo version if you have access)
- [ ] (Optional) Watch Lecture 4: Introduction to RL — preview for Week 2

### 📚 Further Reading
- Ross et al. (2011) "A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning" — the DAgger paper
- CS285 Lecture 2 slides: `rail.eecs.berkeley.edu/deeprlcourse-fa23/deeprlcourse-fa23/static/slides/lec-2.pdf`
- Your UCSD IRL research — revisit with fresh eyes after this week

---
## 🧪 Bonus Challenge: LunarLander with a Trained Expert

CartPole is simple enough for a hand-crafted expert. Let's try a harder environment where we first need to **train** an expert (using PPO from stable-baselines3), then clone it.

This foreshadows Week 6 (PPO) — for now, treat the expert as a black box.

In [ ]:
# Install stable-baselines3 to get a trained expert
!pip install stable-baselines3 -q

from stable_baselines3 import PPO

# Train an expert for LunarLander (takes ~2 min on Colab)
print("Training PPO expert for LunarLander-v2 (this is Week 6 preview!)...")
print("This takes ~2 minutes on Colab GPU.")

lunar_expert = PPO(
    'MlpPolicy', 'LunarLander-v2',
    learning_rate=3e-4, n_steps=1024, batch_size=64,
    n_epochs=4, gamma=0.999, gae_lambda=0.98,
    verbose=0, seed=SEED
)
lunar_expert.learn(total_timesteps=200_000)

# Evaluate expert
def sb3_predict(obs):
    action, _ = lunar_expert.predict(obs, deterministic=True)
    return int(action)

expert_mean, expert_std, _ = evaluate_policy(sb3_predict, 'LunarLander-v2', n_episodes=20)
print(f"\nLunarLander Expert: {expert_mean:.1f} ± {expert_std:.1f}")
print(f"(Good performance is >200, landing is ~250+)")

In [ ]:
# ===========================================================
# BC vs DAgger on LunarLander (harder environment)
# ===========================================================

# LunarLander: 8-dim state, 4 discrete actions
LUNAR_OBS_DIM = 8
LUNAR_ACT_DIM = 4

# Collect expert demos
print("Collecting LunarLander expert demonstrations...")
lunar_states, lunar_actions = collect_demonstrations(
    sb3_predict, 'LunarLander-v2', n_episodes=50, max_steps=1000
)

# Train BC
print("\nTraining Behavioral Cloning on LunarLander...")
print("="*60)
lunar_bc, lunar_bc_hist = train_behavioral_cloning(
    lunar_states, lunar_actions, LUNAR_OBS_DIM, LUNAR_ACT_DIM,
    hidden_dim=128, n_epochs=200, eval_every=50,
    eval_env='LunarLander-v2'
)

# Train DAgger
print("\nTraining DAgger on LunarLander...")
print("="*60)
lunar_dagger, lunar_dagger_hist = train_dagger(
    sb3_predict, 'LunarLander-v2', LUNAR_OBS_DIM, LUNAR_ACT_DIM,
    n_initial_episodes=10, n_dagger_iterations=15,
    n_rollout_episodes=5, max_steps=1000,
    n_train_epochs=50, hidden_dim=128
)

print("\n" + "="*60)
print("FINAL COMPARISON — LunarLander-v2")
print("="*60)
bc_final = evaluate_policy(lunar_bc.get_action, 'LunarLander-v2', n_episodes=30)
dagger_final = evaluate_policy(lunar_dagger.get_action, 'LunarLander-v2', n_episodes=30)
print(f"Expert:  {expert_mean:.1f} ± {expert_std:.1f}")
print(f"BC:      {bc_final[0]:.1f} ± {bc_final[1]:.1f}")
print(f"DAgger:  {dagger_final[0]:.1f} ± {dagger_final[1]:.1f}")

In [ ]:
# Final comparison plot for LunarLander
fig, ax = plt.subplots(figsize=(10, 6))

iters = lunar_dagger_hist['iteration']
means = lunar_dagger_hist['eval_reward_mean']
stds = lunar_dagger_hist['eval_reward_std']

ax.plot(iters, means, color='#6C3483', linewidth=2.5, marker='s',
        markersize=7, label='DAgger', zorder=3)
ax.fill_between(iters, [m-s for m,s in zip(means, stds)],
                [m+s for m,s in zip(means, stds)],
                alpha=0.15, color='#6C3483')

ax.axhline(y=expert_mean, color='#2D8B55', linestyle='--', linewidth=2,
           alpha=0.8, label=f'Expert ({expert_mean:.0f})')
ax.axhline(y=bc_final[0], color='#2E5090', linestyle=':', linewidth=2,
           alpha=0.8, label=f'BC ({bc_final[0]:.0f})')

ax.set_xlabel('DAgger Iteration', fontsize=13)
ax.set_ylabel('Mean Episode Reward', fontsize=13)
ax.set_title('LunarLander-v2: DAgger vs Behavioral Cloning\n'
             'DAgger progressively closes the gap to the expert',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📋 Week 1 Summary Card

### Algorithms Implemented
| Algorithm | Lines of Code | Key Idea |
|-----------|:---:|----------|
| Behavioral Cloning | ~30 | Supervised learning on (state, expert_action) pairs |
| DAgger | ~50 | Iteratively collect data from learned policy, label with expert |

### Key Equations
| Equation | Name | Used In |
|----------|------|--------|
| $\mathcal{L} = -\mathbb{E}[\log \pi_\theta(a^*\|s)]$ | BC Loss (Cross-Entropy) | BC, SFT |
| $\text{Error} \propto \epsilon T^2$ | Compounding Error | BC failure analysis |
| $\mathcal{D} \leftarrow \mathcal{D} \cup \{(s \sim \pi_\theta, \pi^*(s))\}$ | DAgger Aggregation | DAgger |

### Bridge to Future Weeks
| This Week | Future Week | Connection |
|-----------|-------------|------------|
| BC | Wk 11: RLHF | SFT is BC on LLM outputs |
| Distribution shift | Wk 11: RLHF | Why SFT alone isn't enough |
| DAgger (interactive) | Wk 11: Online RLHF | Query human on model's own outputs |
| Expert query cost | Wk 13: DPO | Motivation for offline preference learning |
| Imitation learning | Wk 10: IRL | Learn reward function from demos instead |

---
*Next week: REINFORCE and Policy Gradients — learning without an expert, using only rewards.*

**Total time estimate for this notebook: ~10-12 hours** (including lecture watching and reflection writing)